# Recap
It's the continuation of 14.stage1_unsupervised_finetuning.ipynb. In this section, we will perform instruction finetuning on the model that we finetuned in stage-1.

In [32]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
!pip install -U peft bitsandbytes transformers accelerate

In [34]:
!pip install -U trl  #transformer reinforecement learning

In [35]:
!pip install PyMuPDF

In [36]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

False

In [37]:
# Import necessary libraries
import fitz
import re
import tf_keras as keras
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

In [38]:
# this path is an output from stage#1
# this is the unsupervised-fine tunned model.
base_model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
stage1_ft_model_name = "tinyllama-pharma-unsupervised"
ft_model_name = "tinyllama-pharma-instruction"
models_dir = "/content/drive/MyDrive/models"
checkpoint = "checkpoint-1115"
stage1_model_path = f"{models_dir}/{stage1_ft_model_name}/{checkpoint}"
output_dir = f"{models_dir}/{ft_model_name}"

# Stage2 - Supervised Instruction Finetuning (for Structured Output)

**Stage1 (unsupervised finetuning)** taught the model your domain language. It makes the model aware about the pharama specific terms/vocabularies. Here, the model learns how to predict the next token. But it does not know how to respond to the use instructions (like summerization, q&a, classification, generate-content etc...).

In **Stag2 (supervised instruction finetuning)**, we will teach the model how to generate structured outputs using instructions.

The instructions are structured in Alpaca format:
{
  "instruction": "Explain the mechanism of action of Metformin.", "input": "",
  "output": "Metformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose."
}

### Step1: Load the Unsupervised Finetuned Model

Load the model that we finetuned and saved in Stage1.

In [39]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [40]:
unsupervised_model = AutoModelForCausalLM.from_pretrained(stage1_model_path, device_map="auto")

unsupervised_model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [41]:
# prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
prompt = "Explain the mechanism of action of Metformin."

In [42]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [43]:
outputs = unsupervised_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [44]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain the mechanism of action of Metformin.
Explain how to administer Metformin.
Dosage and duration


### Step2: Load the Instruction Data and Prepare Dataset

In [45]:
from datasets import load_dataset

# dataset = load_dataset("csv", data_files="/content/drive/MyDrive/buildllm/notebooks/finetuning2/pharma_instruction_data.csv",split="train")
dataset = load_dataset("csv", data_files="/content/drive/MyDrive/buildllm/notebooks/finetuning2/essential-drugs-instruction.csv",split="train")
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 315
})

In [46]:
dataset[0]

{'instruction': 'Determine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.',
 'input': 'Amoxicillin, child weight 10 kg, diagnosis tonsillitis.',
 'output': 'The usual dosage for tonsillitis is 25 mg/kg 2 times daily. For a child weighing 10 kg, the dose is 250 mg twice daily. This can be administered as 10 ml of a 125 mg/5 ml suspension twice daily or one 250 mg tablet twice daily[cite: 55, 56].'}

In [47]:
# structure the instruction in Alpaca format and store in a new column called text.
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [48]:
dataset = dataset.map(format_example)

In [49]:
dataset[0]

{'instruction': 'Determine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.',
 'input': 'Amoxicillin, child weight 10 kg, diagnosis tonsillitis.',
 'output': 'The usual dosage for tonsillitis is 25 mg/kg 2 times daily. For a child weighing 10 kg, the dose is 250 mg twice daily. This can be administered as 10 ml of a 125 mg/5 ml suspension twice daily or one 250 mg tablet twice daily[cite: 55, 56].',
 'text': '### Instruction:\nDetermine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.\n### Input:\nAmoxicillin, child weight 10 kg, diagnosis tonsillitis.\n### Response:\nThe usual dosage for tonsillitis is 25 mg/kg 2 times daily. For a child weighing 10 kg, the dose is 250 mg twice daily. This can be administered as 10 ml of a 125 mg/5 ml suspension twice daily or one 250 mg tablet twice daily[cite: 55, 56].'}

In [50]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 315
})

In [51]:
dataset['text'][0]

'### Instruction:\nDetermine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.\n### Input:\nAmoxicillin, child weight 10 kg, diagnosis tonsillitis.\n### Response:\nThe usual dosage for tonsillitis is 25 mg/kg 2 times daily. For a child weighing 10 kg, the dose is 250 mg twice daily. This can be administered as 10 ml of a 125 mg/5 ml suspension twice daily or one 250 mg tablet twice daily[cite: 55, 56].'

In [52]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [53]:
# It tokenizes "text" and stores the token_ids of same text as inputs and outputs(labels).
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [54]:
# Tokenization with Response Masking
# It does the same as tokenize_fn(), but it stores the token_ids post the string "### Response:" in output(labels) while evertyhing in inputs.
# Basically it masks the string prior to "### Response:" in the label column.
# Tokenization with Response Masking
def tokenize_and_mask(example):
    texts = example["text"]

    # If batched=True, texts is a list; otherwise wrap it
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    labels_batch = []
    response_marker = "### Response:"

    for text, input_ids in zip(texts, enc["input_ids"]):
        # Find where '### Response:' starts
        response_start = text.find(response_marker)

        if response_start != -1:
            response_token_start = len(
                tokenizer(text[:response_start], add_special_tokens=False)["input_ids"]
            )
        else:
            response_token_start = 0

        labels = input_ids.copy()
        labels[:response_token_start] = [-100] * response_token_start

        labels_batch.append(labels)

    enc["labels"] = labels_batch

    # If not batched, return single example format
    if len(labels_batch) == 1:
        enc["labels"] = labels_batch[0]

    return enc


In [55]:
#tokenized = dataset.map(tokenize_and_mask, batched=True)
tokenized = dataset.map(tokenize_fn, batched=True)

In [56]:
tokenized

Dataset({
    features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 315
})

In [57]:
print(f"text[0]: {tokenized['text'][0]}")
print(f"token ids of text[0]: {tokenized['input_ids'][0]}")
print(f"target labels of text[0]: {tokenized['labels'][0]}")
# attention_mask is used to pad the sequences in a batch to make them all of same length.
# print(f"attention_mask: {tokenized['attention_mask'][500]}")

text[0]: ### Instruction:
Determine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.
### Input:
Amoxicillin, child weight 10 kg, diagnosis tonsillitis.
### Response:
The usual dosage for tonsillitis is 25 mg/kg 2 times daily. For a child weighing 10 kg, the dose is 250 mg twice daily. This can be administered as 10 ml of a 125 mg/5 ml suspension twice daily or one 250 mg tablet twice daily[cite: 55, 56].
token ids of text[0]: [1, 835, 2799, 4080, 29901, 13, 6362, 837, 457, 278, 1959, 470, 284, 3248, 482, 310, 1913, 2251, 293, 453, 262, 363, 263, 2278, 591, 1141, 292, 29871, 29896, 29900, 12118, 411, 23864, 453, 23448, 29889, 13, 2277, 29937, 10567, 29901, 13, 6833, 2251, 293, 453, 262, 29892, 2278, 7688, 29871, 29896, 29900, 12118, 29892, 24876, 19263, 23864, 453, 23448, 29889, 13, 2277, 29937, 13291, 29901, 13, 1576, 9670, 3248, 482, 363, 23864, 453, 23448, 338, 29871, 29906, 29945, 286, 29887, 29914, 9415, 29871, 29906, 3064, 14218, 29889, 1152, 263

| Parameter        | Meaning                     | Typical Value         | Effect                           |
| ---------------- | --------------------------- | --------------------- | -------------------------------- |
| `task_type`      | Model type (Causal/Seq2Seq) | `CAUSAL_LM`           | Ensures correct integration      |
| `r`              | Rank of LoRA matrix         | 4–16                  | Controls trainable param size    |
| `lora_alpha`     | Scaling factor              | 16–64                 | Balances adaptation strength     |
| `lora_dropout`   | Dropout probability         | 0.05                  | Regularization                   |
| `target_modules` | Which layers to tune        | `["q_proj","v_proj"]` | Trade-off between cost & quality |
| `bias`           | Bias fine-tuning            | `"none"`              | Keep simple                      |


### Step3: Instansiate QLORa Model

In [58]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [59]:
# Insert another lora adapter that would be fine-tunned for the instructions
instruction_model_lora = get_peft_model(unsupervised_model, lora_config)

instruction_model_lora.eval()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

### Step4: Train Model

In [60]:
args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_total_limit=1,
    report_to="none"
)

In [61]:
trainer = Trainer(
    model=instruction_model_lora,
    args=args,
    train_dataset=tokenized,
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [62]:
trainer.train()

Step,Training Loss
10,10.357700
20,4.905100
30,0.950900
40,0.388700
50,0.325800
60,0.264500
70,0.220600
80,0.194400
90,0.175600
100,0.159500


TrainOutput(global_step=200, training_loss=0.9614081484079361, metrics={'train_runtime': 195.5771, 'train_samples_per_second': 8.053, 'train_steps_per_second': 1.023, 'total_flos': 5010834692505600.0, 'train_loss': 0.9614081484079361, 'epoch': 5.0})

### Step4: Evaluate Model

In [65]:
model_path = f"{output_dir}/checkpoint-200"

In [66]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [75]:
prompts = [
    "Determine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.::Amoxicillin, child weight 10 kg, diagnosis tonsillitis.",
    "Identify the contraindications for the use of Aspirin in pain management.::Aspirin (acetylsalicylic acid) for pain/fever.",
    "State the precautions for using oral Sertraline in breast-feeding women.::Sertraline, breast-feeding, precautions."
]

In [81]:
for q in prompts:
    print("Prompt:", q)
    print("\n--- Non-instruction/Unsupervised model ---")
    # print("Input:", q)
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = unsupervised_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )
    print("Output>> ", tokenizer.decode(outputs[0], skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    parts = q.split("::")
    # print("Instruction:", parts[0])
    # print("Input:", parts[1])
    prompt = f"### Instruction:\n{parts[0]}\n### Input:\n{parts[1]}\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = instruction_model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.8,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1
            )
    print("Output>> ", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("="*100, "\n")

Prompt: Determine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.::Amoxicillin, child weight 10 kg, diagnosis tonsillitis.

--- Non-instruction/Unsupervised model ---
Output>>  Determine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.::Amoxicillin, child weight 10 kg, diagnosis tonsillitis.

--- Instruction-tuned model ---
Output>>  ### Instruction:
Determine the correct oral dosage of Amoxicillin for a child weighing 10 kg with tonsillitis.
### Input:
Amoxicillin, child weight 10 kg, diagnosis tonsillitis.
### Response:
Administer 50 mg/kg once daily for 5 days[cite: 2476].

Prompt: Identify the contraindications for the use of Aspirin in pain management.::Aspirin (acetylsalicylic acid) for pain/fever.

--- Non-instruction/Unsupervised model ---
Output>>  Identify the contraindications for the use of Aspirin in pain management.::Aspirin (acetylsalicylic acid) for pain/fever.

--- Instruction-tuned model ---
Output

### Step5: Zip and Store the Model

In [ ]:
import shutil
from pathlib import Path

def zip_model(model_dir, output_zip):
    model_dir = Path(model_dir)
    output_zip = Path(output_zip)

    # shutil.make_archive expects path WITHOUT .zip
    shutil.make_archive(
        base_name=output_zip.with_suffix(""),
        format="zip",
        root_dir=model_dir.parent,
        base_dir=model_dir.name
    )

# # Example
zip_model(
    model_dir=output_dir,
    output_zip=f"{models_dir}/{ft_model_name}.zip"
)